In [1]:
# quitar cuando sea .py
# Limpiamos todas las variables
%reset -f

# Análisis de Predicción de Accidentes Cerebrovasculares

Según la Organización Mundial de la Salud (OMS), el accidente cerebrovascular es la segunda causa de muerte a nivel mundial y es responsable de aproximadamente el 11 % del total de muertes.
Este conjunto de datos se utiliza para predecir si un paciente tiene probabilidades de sufrir un accidente cerebrovascular en función de los parámetros de entrada, como el sexo, la edad, diversas enfermedades y el tabaquismo. Cada fila de los datos proporciona información relevante sobre el paciente.

Para la realización del siguiente modelo se utilizo el Dataset de [ kaggle - Stroke Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset?resource=download)



Integrantes 
------------

| N° SIU | Nombre | Apellido | email |
|--------|---------|----------|--------|
| a1802 | Ezequiel Alejandro | Caamaño | ezecaa@gmail.com |
| a1823 | Luis Alberto | Santamaría Jimenez | santamaria.luigi@gmail.com |

# Modelo de redes neuronales 
- ~~Preparación y carga de datos~~
    - ~~Limpieza de datos: manejo de valores faltantes y eliminación de valores atípicos~~
    - ~~Ingeniería de características: creación de categorías para BMI, edad y niveles de glucosa~~
    - ~~Codificación de variables categóricas: transformación de variables categóricas en numéricas~~
    - ~~Preparación para modelado: división en conjuntos de entrenamiento y prueba, y escalado de características~~
    - ~~Balanceo de clases: aplicación de SMOTE para manejar el desbalance~~
    - ~~Preparación para PyTorch: conversión de datos a tensores de PyTorch y transferencia al dispositivo adecuado~~

- ~~Entrenamiento del modelo~~
- ~~Persistencia del modelos .pkl método joblib~~
- ~~ docker
    - ~~Carga de DataSet en PostgreSQL utilizar las mimas funciones de Preparación de datos~
    - ~~.py Predict del modelo~
    - ~~.py interfaz gráfica del modelo -> borowser web~
    - ~~ ejecutar python desde docker~ 
- 

In [ ]:
import pandas as pd
import numpy as np
import torch # DL
import torch.nn as nn # DL Definición de la Red Neuronal
import torch.optim as optim # DL
import intel_extension_for_pytorch as ipex  # Importar la extensión de Intel
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE # Para balancear el dataset


In [14]:
# Configurar para optimizar dispositivo el funcionamiento de PyTorch
#import torch
if torch.backends.mps.is_available():   # Verificar si hay GPU Apple (MPS)
    device = torch.device("mps")        
    print("Usando Apple Metal (MPS)")    
elif torch.cuda.is_available():         # Verificar si hay GPU nvidia (CUDA)
    device = torch.device("cuda")
    print("Usando CUDA GPU")
else:
    device = torch.device("cpu")        # Verificar si hay GPU Intel disponible (XPU)
    print("Usando CPU")

Usando CPU


No es compatible con python 12 - [ERROR: No se pudo encontrar una versión que satisfaga el requisito intel-extension-for-pytorch==2.1.100 (de las versiones: ninguna) #525](https://github.com/intel/intel-extension-for-pytorch/issues/525)

```python
import torch
import intel_extension_for_pytorch as ipex  # Importar la extensión de Intel pip install intel-extension-for-pytorch

# Configurar dispositivo para PyTorch
if torch.backends.mps.is_available(): # Verificar si hay GPU Apple (MPS)
    device = torch.device("mps")
    print("Usando Apple Metal (MPS)")
elif torch.cuda.is_available():         # Verificar si hay GPU nvidia (CUDA)
    device = torch.device("cuda")
    print("Usando CUDA GPU")
elif hasattr(torch, "xpu") and torch.xpu.is_available():  # Verificar si hay GPU Intel disponible (XPU)
    device = torch.device("xpu")
    print("Usando GPU Intel (XPU)")
else:
    device = torch.device("cpu")  #  Usar CPU si no hay otros dispositivos disponibles
    print("Usando CPU")

## Mover el modelo y los datos al dispositivo seleccionado
#model = model.to(device)
#X_train = X_train.to(device)
#y_train = y_train.to(device)
```

In [ ]:
### funciones de procesamiento de datos   ##


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Realiza la limpieza inicial del conjunto de datos
    """
    df_clean = df.copy()
    
    # Manejo de valores faltantes en BMI
    imputer = SimpleImputer(strategy='median')
    df_clean['bmi'] = imputer.fit_transform(df_clean[['bmi']])
    
    # Eliminar valores atípicos extremos en glucose_level
    q1 = df_clean['avg_glucose_level'].quantile(0.01)
    q3 = df_clean['avg_glucose_level'].quantile(0.99)
    df_clean = df_clean[
        (df_clean['avg_glucose_level'] >= q1) & 
        (df_clean['avg_glucose_level'] <= q3)
    ]
    
    return df_clean


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Realiza la ingeniería de características
    """
    df_engineered = df.copy()
    
    # Crear categorías de BMI
    df_engineered['bmi_category'] = pd.cut(
        df_engineered['bmi'],
        bins=[0, 18.5, 24.9, 29.9, np.inf],
        labels=['Bajo peso', 'Normal', 'Sobrepeso', 'Obeso'],
        right=False # Asegura que 18.5 cae en 'Normal', etc.
    )
    
    # Crear categorías de edad
    df_engineered['age_group'] = pd.cut(
        df_engineered['age'],
        bins=[0, 18, 35, 50, 65, np.inf],
        labels=['<18', '18-35', '36-50', '51-65', '>65'],
        right=False 
    )
    
    # Crear categorías de glucosa
    df_engineered['glucose_category'] = pd.cut(
        df_engineered['avg_glucose_level'],
        bins=[0, 70, 100, 125, np.inf],
        labels=['Bajo', 'Normal', 'Pre-diabetes', 'Diabetes'],
        right=False
    )
    # Convertir las nuevas columnas categóricas a tipo 'category' 
    for col in ['bmi_category', 'age_group', 'glucose_category']:
         if col in df_engineered.columns: # Comprobar si la columna existe después del filtrado
            df_engineered[col] = df_engineered[col].astype('category') # Convierte a tipo category
            

    return df_engineered


def encode_variables(df: pd.DataFrame) -> pd.DataFrame:

    df_encoded = df.copy()
    
    # Codificación de variables categóricas
    categorical_columns = ['gender', 'ever_married', 'work_type', 
                         'Residence_type', 'smoking_status',
                         'bmi_category', 'age_group', 'glucose_category']
    
    for column in categorical_columns:
        le = LabelEncoder()
        df_encoded[column] = le.fit_transform(df_encoded[column])
    
    return df_encoded




# Función para escalar y dividir datos ver como mejorar el code donde solo realice el escalado   
def prepare_for_modeling(df: pd.DataFrame, 
                        target: str = 'stroke',
                        test_size: float = 0.2,
                        random_state: int = 42) -> tuple:
    """
    Prepara los datos para el modelado: separa X/y, divide train/test, y escala X.
    """
    # Separar características y objetivo
    X = df.drop(columns=[target])
    y = df[target]
    
    # División train-test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    ## Escalado de características
    # Escalado de características numéricas
    # Identificar columnas numéricas para escalar (excluir las ya codificadas si son el resultado final)
    # Si usamos LabelEncoder en todo, todas las X serán numéricas en este punto.
    numeric_cols = X_train.select_dtypes(include=np.number).columns # Asumiendo que todas son numéricas tras encode

    scaler = StandardScaler()
    # Ajustar y transformar en el conjunto de entrenamiento
    X_train_scaled = scaler.fit_transform(X_train[numeric_cols])
    # Solo transformar en el conjunto de prueba (usando el ajuste del entrenamiento)
    X_test_scaled = scaler.transform(X_test[numeric_cols])

    # Convertir de nuevo a DataFrame para mantener nombres de columnas (opcional pero útil)
    X_train_scaled = pd.DataFrame(X_train_scaled, index=X_train.index, columns=numeric_cols)
    X_test_scaled = pd.DataFrame(X_test_scaled, index=X_test.index, columns=numeric_cols)

    # Asegurarse de que y_train, y_test sean Series de Pandas para el paso de SMOTE
    if not isinstance(y_train, pd.Series): y_train = pd.Series(y_train, index=X_train.index)
    if not isinstance(y_test, pd.Series): y_test = pd.Series(y_test, index=X_test.index)


    return X_train_scaled, X_test_scaled, y_train, y_test, scaler
"""    
    ## Escalado de características
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler
"""



### Aplica SMOTE para balancear el conjunto de datos de entrenamiento.

"""

def balance_dataset(X_train, y_train, random_state=42):
    # Manejo del desbalance de clases con SMOTE
    smote = SMOTE(random_state=random_state)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    return X_train_balanced, y_train_balanced


""" 

def balance_dataset(X_train, y_train, random_state=42):

    ### Aplica SMOTE para balancear el conjunto de datos de entrenamiento.

    print(f"def balance_dataset - Distribución antes de SMOTE:\n{y_train.value_counts(normalize=True)}")
    # Manejo del desbalance de clases con SMOTE
    smote = SMOTE(random_state=random_state)
    # SMOTE espera arrays de numpy generalmente
    X_train_np = X_train.to_numpy() if isinstance(X_train, pd.DataFrame) else X_train
    y_train_np = y_train.to_numpy() if isinstance(y_train, pd.Series) else y_train

    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_np, y_train_np)
    print(f"def balance_dataset - Forma después de SMOTE: {X_train_balanced.shape}")
    print(f"def balance_dataset - Distribución después de SMOTE:\n{pd.Series(y_train_balanced).value_counts(normalize=True)}")
    return X_train_balanced, y_train_balanced




In [5]:
# --- Definición de la Red Neuronal feedforward - clasificación binaria---
class FeedForwardNN(nn.Module):
    def __init__(self, input_size, hidden_sizes=[128, 64, 32]):
        super().__init__()
        layers = []
        prev_size = input_size

        # Capas ocultas con normalización por lotes y activación mejorada
        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.BatchNorm1d(hidden_size), # BatchNorm ayuda a estabilizar
                nn.ReLU(),
                nn.Dropout(0.3) # Dropout para regularización
            ])
            prev_size = hidden_size

        # Capa de salida
        layers.append(nn.Linear(prev_size, 1)) # Salida única para clasificación binaria
        # No añadimos Sigmoid aquí, usaremos BCEWithLogitsLoss que es más estable 
        # layers.append(nn.Sigmoid()) # si quitamos esta linea el BCEWithLogitsLoss esta por defecto 

        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

In [6]:
# --- Función para preparar datos para PyTorch ---
def prepare_data(X_train, X_test, y_train, y_test):
    """
    Prepara los datos (NumPy arrays) para PyTorch y los mueve al dispositivo.
    """
    # Configurar dispositivo para PyTorch
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Usando Apple Metal (MPS)")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("Usando CUDA GPU")
    else:
        device = torch.device("cpu")
        print("Usando CPU")

    # Convertir DataFrames/Series a arreglos numpy si aún no lo son
    if isinstance(X_train, pd.DataFrame): X_train = X_train.to_numpy()
    if isinstance(X_test, pd.DataFrame): X_test = X_test.to_numpy()
    if isinstance(y_train, pd.Series): y_train = y_train.to_numpy()
    if isinstance(y_test, pd.Series): y_test = y_test.to_numpy()

    # Asegurar que X_train y X_test sean bidimensionales (ya deberían serlo tras StandardScaler)
    if len(X_train.shape) == 1: X_train = X_train.reshape(-1, 1)
    if len(X_test.shape) == 1: X_test = X_test.reshape(-1, 1)

    # Convertir a Tensores de PyTorch (usar FloatTensor para X, podría ser LongTensor para y si la loss lo requiere, pero BCEWithLogitsLoss prefiere Float)
    X_train = torch.FloatTensor(X_train).to(device)
    X_test = torch.FloatTensor(X_test).to(device)
    y_train = torch.FloatTensor(y_train).reshape(-1, 1).to(device) # BCEWithLogitsLoss espera y como Float
    y_test = torch.FloatTensor(y_test).reshape(-1, 1).to(device)

    return X_train, X_test, y_train, y_test, device


#### --- Flujo inicio ---

In [7]:

# 1. Carga de datos
DATA_PATH = "data/healthcare-dataset-stroke-data.csv" # Asegúrate que la ruta sea correcta
# df = pd.read_csv(DATA_PATH)
try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en {DATA_PATH}")
    exit() # Salir si no se encuentra el archivo


In [8]:
# 2. Preprocesamiento
df_clean = clean_data(df)
df_engineered = engineer_features(df_clean)
df_encoded = encode_variables(df_engineered) # Guardamos los encoders por si acaso

# 3. Preparación para Modelado (Divide y Escala)
X_train_scaled, X_test_scaled, y_train, y_test, scaler = prepare_for_modeling(df_encoded)

# 4. Balanceo del Dataset de Entrenamiento (SMOTE)
#    Asegúrate que X_train_scaled es NumPy array o compatible con SMOTE
X_train_balanced, y_train_balanced = balance_dataset(X_train_scaled, y_train)

# 5. Preparación para PyTorch (Convertir a Tensores y mover a Dispositivo) función prepare_data
X_train_tensor, X_test_tensor, y_train_tensor, y_test_tensor, device = prepare_data(
    X_train_balanced, X_test_scaled.to_numpy(), y_train_balanced, y_test.to_numpy() # Asegurar que X_test es numpy
)



def balance_dataset - Distribución antes de SMOTE:
stroke
0    0.952298
1    0.047702
Name: proportion, dtype: float64
def balance_dataset - Forma después de SMOTE: (7626, 14)
def balance_dataset - Distribución después de SMOTE:
0    0.5
1    0.5
Name: proportion, dtype: float64
Usando CPU


In [9]:
# --- Configuración del Entrenamiento ---

# Hiperparámetros
INPUT_SIZE = X_train_tensor.shape[1] # Número de características tras preprocesamiento
HIDDEN_SIZES = [128, 64, 32] # Arquitectura definida en la clase
OUTPUT_SIZE = 1 # Salida única para clasificación binaria
LEARNING_RATE = 0.001
EPOCHS = 100 # Número de veces que se itera sobre todo el dataset
BATCH_SIZE = 64 # Tamaño de los lotes de datos para entrenar

# Crear DataLoaders para manejar los lotes eficientemente
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False) # No barajar en test

# Instanciar el modelo y moverlo al dispositivo
model = FeedForwardNN(input_size=INPUT_SIZE, hidden_sizes=HIDDEN_SIZES).to(device)

# Función de Pérdida y Optimizador
# Usar BCEWithLogitsLoss es numéricamente más estable que Sigmoid + BCELoss
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)



# --- Bucle de Entrenamiento ---
print("\n--- Iniciando Entrenamiento ---")
for epoch in range(EPOCHS):
    model.train() # Poner el modelo en modo entrenamiento (activa Dropout, BatchNorm en modo train)
    running_loss = 0.0

    for i, (inputs, labels) in enumerate(train_loader):
        # Los DataLoaders ya deberían tener los tensores en el dispositivo correcto
        # si se crearon a partir de tensores ya movidos. Si no:
        # inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels) # Calcular la pérdida

        # Backward pass y optimización
        optimizer.zero_grad() # Limpiar gradientes anteriores
        loss.backward() # Calcular gradientes
        optimizer.step() # Actualizar pesos

        running_loss += loss.item()

    # Imprimir pérdida promedio de la época
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {epoch_loss:.4f}")

    # --- Evaluación Opcional por Época (en el conjunto de test) ---
    if (epoch + 1) % 10 == 0: # Evaluar cada 10 épocas, por ejemplo
        model.eval() # Poner el modelo en modo evaluación (desactiva Dropout, BatchNorm usa estadísticas acumuladas)
        test_loss = 0.0
        correct = 0
        total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad(): # Desactivar cálculo de gradientes para evaluación
            for inputs, labels in test_loader:
                # inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()

                # Convertir logits a probabilidades (aplicando Sigmoid) y luego a predicciones (0 o 1)
                predicted = torch.sigmoid(outputs) > 0.5
                # predicted = (outputs > 0).float() # Alternativa si se usa BCEWithLogitsLoss (logits > 0 equivale a sigmoid > 0.5)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                # Guardar para reporte de clasificación
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())


        avg_test_loss = test_loss / len(test_loader)
        accuracy = 100 * correct / total


        
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Evaluación Test: Loss: {avg_test_loss:.4f}, Accuracy: {accuracy:.2f}%")
        # print(classification_report(all_labels, all_preds, target_names=['No Stroke', 'Stroke']))


print("\n--- Entrenamiento Finalizado ---")



--- Iniciando Entrenamiento ---
Epoch [1/100], Loss: 0.5202
Epoch [2/100], Loss: 0.4451
Epoch [3/100], Loss: 0.4221
Epoch [4/100], Loss: 0.4048
Epoch [5/100], Loss: 0.3867
Epoch [6/100], Loss: 0.3753
Epoch [7/100], Loss: 0.3642
Epoch [8/100], Loss: 0.3520
Epoch [9/100], Loss: 0.3452
Epoch [10/100], Loss: 0.3432
Epoch [10/100] - Evaluación Test: Loss: 0.4423, Accuracy: 74.85%
Epoch [11/100], Loss: 0.3372
Epoch [12/100], Loss: 0.3233
Epoch [13/100], Loss: 0.3208
Epoch [14/100], Loss: 0.3271
Epoch [15/100], Loss: 0.3200
Epoch [16/100], Loss: 0.3127
Epoch [17/100], Loss: 0.3030
Epoch [18/100], Loss: 0.2957
Epoch [19/100], Loss: 0.2960
Epoch [20/100], Loss: 0.2887
Epoch [20/100] - Evaluación Test: Loss: 0.3872, Accuracy: 79.24%
Epoch [21/100], Loss: 0.2875
Epoch [22/100], Loss: 0.2815
Epoch [23/100], Loss: 0.2807
Epoch [24/100], Loss: 0.2741
Epoch [25/100], Loss: 0.2767
Epoch [26/100], Loss: 0.2630
Epoch [27/100], Loss: 0.2672
Epoch [28/100], Loss: 0.2688
Epoch [29/100], Loss: 0.2535
Epoch

---

In [ ]:
# --- Evaluación Final ---
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        # inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        # Convertir logits a predicciones
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n--- Reporte de Clasificación Final (Test Set) ---")
print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=['No Stroke (0)', 'Stroke (1)']))
final_accuracy = accuracy_score(all_labels, all_preds)
print(f"Accuracy Final en Test: {final_accuracy * 100:.2f}%")




--- Reporte de Clasificación Final (Test Set) ---
[[845 109]
 [ 30  18]]
               precision    recall  f1-score   support

No Stroke (0)       0.97      0.89      0.92       954
   Stroke (1)       0.14      0.38      0.21        48

     accuracy                           0.86      1002
    macro avg       0.55      0.63      0.56      1002
 weighted avg       0.93      0.86      0.89      1002

Accuracy Final en Test: 86.13%




---

# Guardado de modelo - persistencia de modelo 

Se puede guardar tu modelo entrenado de PyTorch usando joblib.dump, aunque la forma estándar y más recomendada en PyTorch es guardar el state_dict del modelo, no el objeto del modelo completo



## Método 1: Guardar solo el state_dict (Recomendado)

El state_dict es un diccionario de Python que contiene todos los pesos y sesgos (parámetros entrenables) del modelo. Es la forma más flexible y robusta de guardar modelos en PyTorch, ya que desacopla los pesos guardados de la estructura exacta del código de la clase del modelo

joblib utiliza el módulo pickle de Python internamente, por lo que las consideraciones son similares.

In [11]:
#############    mÉtodo standard torch.save SIN joblib
# torch.save(model.state_dict(), 'MLcerebrovascular.pth')
# print("Modelo guardado en stroke_ffnn_model.pth")

In [12]:
# code mas elaborado torch.save
import joblib
import torch # Asegúrate de que torch esté importado

# --- Asumiendo que tu modelo entrenado está en la variable 'model' ---
# --- y que ya has terminado el entrenamiento ---

# 1. Obtener el state_dict del modelo
#    Es importante mover los parámetros a la CPU ANTES de guardarlos
#    si planeas cargarlos en un entorno que podría no tener la misma GPU o ninguna.
model.cpu() # Mueve el modelo y sus parámetros a la CPU
state_dict_to_save = model.state_dict()

# 2. Definir el nombre del archivo
filename_statedict = 'MLcerebrovascular.pkl'

# 3. Guardar el state_dict usando joblib
try:
    joblib.dump(state_dict_to_save, filename_statedict)
    print(f"Estado del modelo (state_dict) guardado exitosamente en: {filename_statedict}")
except Exception as e:
    print(f"Error al guardar el state_dict: {e}")

# (Opcional) Mover el modelo de nuevo al dispositivo original si continúas usándolo
# model.to(device) # 'device' debe ser la variable definida anteriormente (cuda, mps, cpu)

# --- Para cargar el state_dict más tarde ---
# Necesitas tener la definición de tu clase FeedForwardNN disponible

# a. Primero, instancia la estructura del modelo (con los mismos hiperparámetros)
#    INPUT_SIZE debe ser el mismo que usaste para entrenar
input_size_for_loading = X_train_tensor.shape[1] # O el valor que corresponda
loaded_model_structure = FeedForwardNN(input_size=input_size_for_loading, hidden_sizes=HIDDEN_SIZES)

# b. Carga el state_dict desde el archivo .pkl
try:
    loaded_state_dict = joblib.load(filename_statedict)
    print(f"State_dict cargado desde: {filename_statedict}")

    # c. Carga el state_dict en la estructura del modelo
    loaded_model_structure.load_state_dict(loaded_state_dict)
    print("Pesos cargados exitosamente en la estructura del modelo.")

    # d. ¡Importante! Poner el modelo en modo evaluación si vas a hacer inferencia
    loaded_model_structure.eval()

    # Ahora puedes usar loaded_model_structure para hacer predicciones
    # Ejemplo:
    # with torch.no_grad():
    #     prediction = loaded_model_structure(un_nuevo_tensor_de_entrada)

except Exception as e:
    print(f"Error al cargar el state_dict o los pesos: {e}")

Estado del modelo (state_dict) guardado exitosamente en: MLcerebrovascular.pkl
State_dict cargado desde: MLcerebrovascular.pkl
Pesos cargados exitosamente en la estructura del modelo.
